# Importing Libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import f1_score, classification_report

# Importing the Dataset

In [2]:
train_data = pd.read_csv("/content/Corona_NLP_train.csv", encoding='latin-1')
test_data = pd.read_csv("/content/Corona_NLP_test.csv", encoding='latin-1')

# Preprocessing Dataset

In [3]:
train_data = train_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)
test_data = test_data.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis=1)

In [4]:
le = preprocessing.LabelEncoder()
le.fit(train_data['Sentiment'])
train_data['Sentiment'] = le.transform(train_data['Sentiment'])
test_data['Sentiment'] = le.transform(test_data['Sentiment'])

In [5]:
X_train = train_data['OriginalTweet']
y_train = train_data['Sentiment']
X_test = test_data['OriginalTweet']
y_test = test_data['Sentiment']

In [6]:
y_train = to_categorical(y_train, num_classes=5)
y_test = to_categorical(y_test, num_classes=5)

# Compute text Statistics for Vectorization

In [7]:
averageWordsLength = round(sum([len(i.split()) for i in train_data['OriginalTweet']]) / len(train_data['OriginalTweet']))
totalWordsLength = len(set(" ".join(train_data['OriginalTweet']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 41157
Average words per message: 31
Approximate vocabulary size: 136386


# Model development

In [9]:
model = keras.Sequential()
model.add(layers.Input(shape=(1,), dtype=tf.string))

In [10]:
text_vectorizer = layers.TextVectorization(
    max_tokens=20000,#totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=50#averageWordsLength
)
text_vectorizer.adapt(X_train)

model.add(text_vectorizer)

In [14]:
vocab_size = text_vectorizer.get_vocabulary()
word_index = dict(zip(vocab_size, range(len(vocab_size))))
embedding_index = {}

with open('glove.6B.50d.txt', 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = coefs

embedding_dim=50
embedding_matrix = np.zeros((len(vocab_size), embedding_dim))

for word, i in word_index.items():
    if i < len(vocab_size):
        embedding_vector = embedding_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

In [16]:
embedding_layer = layers.Embedding(
    input_dim=20000,#totalWordsLength,
    output_dim=embedding_dim,
    trainable=False,
    weights=[embedding_matrix]
)
model.add(embedding_layer)

In [17]:
model.add(layers.LSTM(64, dropout=0.3, recurrent_dropout=0.3))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(5, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 50)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 50, 50)         │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        29,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,033,925 (3.94 MB)

 Trainable params: 33,925 (132.52 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

# Model training

In [18]:
history = model.fit(X_train.to_numpy(), y_train, epochs=20, batch_size=64, validation_data=(X_test.to_numpy(), y_test))

Epoch 1/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 42s 59ms/step - accuracy: 0.2907 - loss: 1.5345 - val_accuracy: 0.3741 - val_loss: 1.4113
Epoch 2/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 39s 55ms/step - accuracy: 0.3727 - loss: 1.4216 - val_accuracy: 0.4186 - val_loss: 1.3282
Epoch 3/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 40s 54ms/step - accuracy: 0.4144 - loss: 1.3398 - val_accuracy: 0.4373 - val_loss: 1.2896
Epoch 4/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 43s 57ms/step - accuracy: 0.4427 - loss: 1.2934 - val_accuracy: 0.4579 - val_loss: 1.2312
Epoch 5/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 36s 56ms/step - accuracy: 0.4512 - loss: 1.2690 - val_accuracy: 0.4710 - val_loss: 1.2213
Epoch 6/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 35s 54ms/step - accuracy: 0.4605 - loss: 1.2457 - val_accuracy: 0.4842 - val_loss: 1.1920
Epoch 7/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 44s 58ms/step - accuracy: 0.4624 - loss: 1.2289 - val_accuracy: 0.4950 - val_loss: 1.1836
Epoch 8/20
644/644 ━━━━━━━━━━━━━━━━━━━━ 36s 56ms/step - accuracy: 0.4797 - loss: 1.2129 - 

# Model Prediction

In [25]:
y_pred = model.predict(X_test.to_numpy())
y_pred = np.argmax(y_pred, axis=1)
y_test = np.argmax(y_test, axis=1)

In [24]:
print(classification_report(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

           0       0.65      0.63      0.64       592
           1       0.67      0.58      0.62       599
           2       0.52      0.53      0.53      1041
           3       0.66      0.64      0.65       619
           4       0.47      0.51      0.49       947

    accuracy                           0.57      3798
   macro avg       0.59      0.58      0.58      3798
weighted avg       0.57      0.57      0.57      3798

0.5695036430562238
